In [3]:
# NN2Logic — Scaled Deeper/Wider Network
# Extends the XOR prototype to support:
#   - Multiple hidden layers (deeper)
#   - More neurons per layer (wider)
#   - Any binary classification dataset
# All pipeline stages remain identical — only the network config changes.

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import tensorflow as tf
from tensorflow import keras

# ============================================================
# VISUALIZATION SETUP — uses matplotlib + networkx (no Graphviz binary needed)
# ============================================================
import matplotlib
matplotlib.use("Agg")          # safe for headless / Colab / scripts
import matplotlib.pyplot as plt

try:
    import networkx as nx
    NX_AVAILABLE = True
except ImportError:
    NX_AVAILABLE = False
    print("[VIZ WARNING] networkx not installed — graph visualizations will be skipped.")
    print("              Install with: pip install networkx")

VIZ_DIR = "viz_output"
os.makedirs(VIZ_DIR, exist_ok=True)
print(f"[VIZ] output folder: {os.path.abspath(VIZ_DIR)}")


def _draw_dag(nodes_data, edges, filename, title=""):
    """Generic DAG renderer using matplotlib + networkx."""
    if not NX_AVAILABLE:
        return
    G = nx.DiGraph()
    labels = {}
    for nid, lbl, _col in nodes_data:
        G.add_node(nid)
        labels[nid] = lbl
    for s, d in edges:
        G.add_edge(s, d)

    # Topological layered layout (sources at bottom, sinks at top)
    try:
        layers = list(nx.topological_generations(G))
    except nx.NetworkXUnfeasible:
        layers = [list(G.nodes())]

    pos = {}
    for layer_idx, layer in enumerate(layers):
        for i, n in enumerate(layer):
            pos[n] = (i - len(layer) / 2, layer_idx)

    color_map = {nid: col for nid, _, col in nodes_data}
    node_colors = [color_map.get(n, "white") for n in G.nodes()]

    fig_w = max(6, len(G.nodes()) * 0.45)
    fig_h = max(4, len(layers) * 1.2)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=True,
                           arrowsize=14, edge_color='gray', width=1.2)
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=node_colors, node_size=900,
                           edgecolors='black', linewidths=1)
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=9)
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    full_path = os.path.join(VIZ_DIR, filename + ".png")
    plt.savefig(full_path, dpi=120, bbox_inches='tight')
    plt.close()


def visualize_ac(ac, filename, title=""):
    colour = {"INPUT": "#9ecae1", "CONST": "#fff7bc",
              "MUL":   "#a1d99b", "ADD":   "#fc9272",
              "ACT":   "#fdae6b"}
    nodes_data, edges = [], []
    for n in ac.nodes:
        if   n.op == "INPUT": label = f"{n.value}"
        elif n.op == "CONST": label = f"{n.value:.3f}"
        elif n.op == "ACT":   label = "STEP"
        else:                 label = n.op
        nodes_data.append((str(n.id), label, colour.get(n.op, "white")))
        for c in n.inputs:
            edges.append((str(c.id), str(n.id)))
    _draw_dag(nodes_data, edges, filename, title)


def visualize_nnf(nnf, filename, title=""):
    nodes_data, edges = [], []
    for n in nnf.nodes:
        if n.op == "LITERAL":
            sign  = "" if n.literal > 0 else "¬"
            label = f"{sign}x{abs(n.literal)}"
            col   = "#9ecae1"
        elif n.op == "AND":
            label, col = "∧", "#a1d99b"
        elif n.op == "OR":
            label, col = "∨", "#fc9272"
        else:
            label, col = n.op, "white"
        nodes_data.append((str(n.id), label, col))
        for c in n.inputs:
            edges.append((str(c.id), str(n.id)))
    _draw_dag(nodes_data, edges, filename, title)


def plot_training_curve(history, filename="training_curve_multilayer.png"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(history.history['loss'], color='tab:red')
    ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss"); ax1.grid(True, alpha=0.3)
    ax2.plot(history.history['accuracy'], color='tab:green')
    ax2.set_title("Training Accuracy"); ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy"); ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    full_path = os.path.join(VIZ_DIR, filename)
    plt.savefig(full_path, dpi=120)
    plt.close()
    print(f"[VIZ] wrote {full_path}")


print("NN2Logic — Scaled Network Experiment")

# ============================================================
# NETWORK CONFIGURATION — change here only
# ============================================================

DATASET = "xor"          # "xor" | "and" | "or" | "majority3"

HIDDEN_LAYERS  = [8, 4]
MAX_EPOCHS     = 10000
LEARNING_RATE  = 0.1
MOMENTUM       = 0.9
PATIENCE       = 200
MAX_ATTEMPTS   = 10
SCALE_FACTORS  = [1000, 10000, 100000]

# ============================================================
# DATASET DEFINITIONS
# ============================================================

def get_dataset(name):
    if name == "xor":
        X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
        y = np.array([0,1,1,0],                dtype=np.float32)
    elif name == "and":
        X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
        y = np.array([0,0,0,1],                dtype=np.float32)
    elif name == "or":
        X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float32)
        y = np.array([0,1,1,1],                dtype=np.float32)
    elif name == "majority3":
        X = np.array([[0,0,0],[0,0,1],[0,1,0],[0,1,1],
                      [1,0,0],[1,0,1],[1,1,0],[1,1,1]], dtype=np.float32)
        y = np.array([0,0,0,1,0,1,1,1],                dtype=np.float32)
    else:
        raise ValueError(f"Unknown dataset: {name}")
    return X, y

X_train, y_train = get_dataset(DATASET)
n_inputs         = X_train.shape[1]
print(f"\nDataset      : {DATASET}")
print(f"Inputs       : {n_inputs}")
print(f"Samples      : {len(X_train)}")
print(f"Hidden layers: {HIDDEN_LAYERS}")

# ============================================================
# AC AND NNF DATA STRUCTURES
# ============================================================

class ACNode:
    def __init__(self, id, op, inputs=None, value=None):
        self.id, self.op, self.inputs, self.value = id, op, inputs or [], value

class NNFNode:
    def __init__(self, id, op, inputs=None, literal=None):
        self.id, self.op, self.inputs, self.literal = id, op, inputs or [], literal

class AC:
    def __init__(self):
        self.nodes, self.count, self.output = [], 0, None
    def add(self, op, **kw):
        n = ACNode(f"n{self.count+1}", op, **kw)
        self.nodes.append(n); self.count += 1
        return n

class NNF:
    def __init__(self, n_vars):
        self.nodes, self.count, self.output, self.n_vars = [], 0, None, n_vars
    def add(self, op, **kw):
        n = NNFNode(self.count, op, **kw)
        self.nodes.append(n); self.count += 1
        return n

# ============================================================
# AC EVALUATOR
# ============================================================

def eval_ac(ac, vals):
    v = {}
    for n in ac.nodes:
        if   n.op == "INPUT": v[n.id] = vals[n.value]
        elif n.op == "CONST": v[n.id] = n.value
        elif n.op == "MUL":   v[n.id] = v[n.inputs[0].id] * v[n.inputs[1].id]
        elif n.op == "ADD":   v[n.id] = sum(v[i.id] for i in n.inputs)
        elif n.op == "ACT":   v[n.id] = 1 if v[n.inputs[0].id] > 0 else 0
    return v[ac.output.id]

def eval_nnf(nnf, vals, var_names):
    v = {}
    for n in nnf.nodes:
        if n.op == "LITERAL":
            idx     = abs(n.literal) - 1
            val     = vals[var_names[idx]]
            v[n.id] = val if n.literal > 0 else 1 - val
        elif n.op == "AND":
            v[n.id] = 1 if all(v[i.id] for i in n.inputs) else 0
        elif n.op == "OR":
            v[n.id] = 1 if any(v[i.id] for i in n.inputs) else 0
    return v[nnf.output.id] if nnf.output is not None else 0

# ============================================================
# OBDD PSEUDO-POLYNOMIAL ALGORITHM
# ============================================================

def _to_integers(weights, bias, scale_override=None):
    if scale_override is not None:
        return ([int(round(w * scale_override)) for w in weights],
                int(round(bias * scale_override)))
    all_vals   = [abs(w) for w in weights] + [abs(bias)]
    non_zero   = [v for v in all_vals if v > 1e-9]
    safe_scale = max(1000, int(100.0 / min(non_zero)) + 1) if non_zero else 1000
    return ([int(round(w * safe_scale)) for w in weights],
            int(round(bias * safe_scale)))

def compile_neuron_obdd(weights, bias, var_names, scale_override=None):
    n            = len(weights)
    int_w, int_b = _to_integers(weights, bias, scale_override)
    states       = {int_b: [[]]}
    for i, w in enumerate(int_w):
        next_states = {}
        for acc_sum, paths in states.items():
            for path in paths:
                next_states.setdefault(acc_sum,     []).append(path + [-(i+1)])
                next_states.setdefault(acc_sum + w, []).append(path + [ (i+1)])
        states = next_states
    nnf = NNF(n)
    and_terms = []
    for acc_sum, paths in states.items():
        if acc_sum > 0:
            for path in paths:
                lits = [nnf.add("LITERAL", literal=lit) for lit in path]
                and_terms.append(nnf.add("AND", inputs=lits))
    nnf.output = nnf.add("OR", inputs=and_terms)
    return nnf

def compile_neuron_obdd_verified(weights, bias, var_names):
    ac_ref  = build_ac_for_neuron(weights, bias, var_names)
    all_inp = get_all_binary_inputs(len(var_names))
    ac_sat  = sum(1 for inp in all_inp
                  if eval_ac(ac_ref, dict(zip(var_names, inp))) == 1)
    for scale in SCALE_FACTORS:
        nnf     = compile_neuron_obdd(weights, bias, var_names, scale_override=scale)
        nnf_sat = sum(1 for inp in all_inp
                      if eval_nnf(nnf, dict(zip(var_names, inp)), var_names) == 1)
        if nnf_sat == ac_sat:
            return nnf
        print(f"    [scale={scale}] NNF sat={nnf_sat} != AC sat={ac_sat}, retrying...")
    raise RuntimeError(
        f"OBDD compilation failed for weights={weights}, bias={bias}. "
        f"AC sat={ac_sat}. Try adding larger scale to SCALE_FACTORS."
    )

# ============================================================
# NNF COMPOSITION
# ============================================================

def _copy_nnf_into(src_node, src_nnf, dst_nnf, memo):
    key = (id(src_nnf), src_node.id)
    if key in memo: return memo[key]
    if src_node.op == "LITERAL":
        new_node = dst_nnf.add("LITERAL", literal=src_node.literal)
    elif src_node.op in ("AND", "OR"):
        copied = [_copy_nnf_into(c, src_nnf, dst_nnf, memo) for c in src_node.inputs]
        new_node = dst_nnf.add(src_node.op, inputs=copied)
    else:
        new_node = dst_nnf.add(src_node.op)
    memo[key] = new_node
    return new_node

def _negate_nnf_node(nnf_out, node, src_nnf, memo):
    key = (id(src_nnf), node.id)
    if key in memo: return memo[key]
    if node.op == "LITERAL":
        new_node = nnf_out.add("LITERAL", literal=-node.literal)
    elif node.op == "AND":
        neg_ch = [_negate_nnf_node(nnf_out, c, src_nnf, memo) for c in node.inputs]
        new_node = nnf_out.add("OR", inputs=neg_ch)
    elif node.op == "OR":
        neg_ch = [_negate_nnf_node(nnf_out, c, src_nnf, memo) for c in node.inputs]
        new_node = nnf_out.add("AND", inputs=neg_ch)
    else:
        new_node = node
    memo[key] = new_node
    return new_node

def compose_nnfs(output_nnf, hidden_nnfs, n_original_vars):
    composed         = NNF(n_original_vars)
    shared_copy_memo = {}
    shared_neg_memo  = {}
    hidden_roots     = {}
    hidden_neg_roots = {}
    for idx, h_nnf in enumerate(hidden_nnfs):
        all_inp     = get_all_binary_inputs(h_nnf.n_vars)
        var_names_h = [f"x{i+1}" for i in range(h_nnf.n_vars)]
        sat = sum(1 for inp in all_inp
                  if eval_nnf(h_nnf, dict(zip(var_names_h, inp)), var_names_h) == 1)
        if sat == len(all_inp):
            hidden_roots[idx]     = composed.add("AND", inputs=[])
            hidden_neg_roots[idx] = composed.add("OR",  inputs=[])
        elif sat == 0:
            hidden_roots[idx]     = composed.add("OR",  inputs=[])
            hidden_neg_roots[idx] = composed.add("AND", inputs=[])
        else:
            hidden_roots[idx]     = _copy_nnf_into(h_nnf.output, h_nnf, composed, shared_copy_memo)
            hidden_neg_roots[idx] = _negate_nnf_node(composed, hidden_roots[idx], h_nnf, shared_neg_memo)

    def substitute(node, memo):
        if node.id in memo: return memo[node.id]
        if node.op == "LITERAL":
            hi_idx = abs(node.literal) - 1
            new_node = (hidden_roots[hi_idx] if node.literal > 0
                        else hidden_neg_roots[hi_idx]) \
                       if hi_idx < len(hidden_nnfs) \
                       else composed.add("LITERAL",
                                         literal=(1 if node.literal > 0 else -1) * (hi_idx+1))
        elif node.op in ("AND", "OR"):
            new_node = composed.add(node.op,
                                    inputs=[substitute(c, memo) for c in node.inputs])
        else:
            new_node = composed.add(node.op)
        memo[node.id] = new_node
        return new_node
    composed.output = substitute(output_nnf.output, {})
    return composed

# ============================================================
# HELPERS
# ============================================================

def get_all_binary_inputs(n):
    if n > 20:
        raise ValueError(f"Exhaustive enumeration for n={n} = {2**n} combinations.")
    if n > 10:
        print(f"  [WARNING] n={n} inputs → {2**n} combinations.")
    return [[int(b) for b in format(i, f'0{n}b')] for i in range(2**n)]

def build_ac_for_neuron(weights, bias, var_names):
    ac           = AC()
    input_nodes  = [ac.add("INPUT", value=v) for v in var_names]
    weight_nodes = [ac.add("CONST", value=float(w)) for w in weights]
    bias_node    = ac.add("CONST", value=float(bias))
    mul_nodes    = [ac.add("MUL", inputs=[input_nodes[i], weight_nodes[i]])
                    for i in range(len(var_names))]
    s            = ac.add("ADD", inputs=mul_nodes + [bias_node])
    ac.output    = ac.add("ACT", inputs=[s], value="STEP")
    return ac

def extract_layer_weights(model):
    return [{'name': l.name, 'n_inputs': l.get_weights()[0].shape[0],
             'n_neurons': l.get_weights()[0].shape[1],
             'W': l.get_weights()[0], 'b': l.get_weights()[1]}
            for l in model.layers if isinstance(l, keras.layers.Dense)]

# ============================================================
# BUILD MODEL
# ============================================================

tf.random.set_seed(0); np.random.seed(0)
layers_list = [keras.Input(shape=(n_inputs,), name="input_layer")]
for i, n_neurons in enumerate(HIDDEN_LAYERS):
    layers_list.append(keras.layers.Dense(
        n_neurons, activation='tanh', use_bias=True,
        kernel_initializer='glorot_uniform', name=f'hidden_layer_{i+1}'))
layers_list.append(keras.layers.Dense(1, activation='sigmoid', use_bias=True, name='output_layer'))

model = keras.Sequential(layers_list, name='nn2logic_scaled')
model.compile(optimizer=keras.optimizers.SGD(learning_rate=LEARNING_RATE, momentum=MOMENTUM),
              loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
print(f"\nTotal hidden layers : {len(HIDDEN_LAYERS)}")
print(f"Neurons per layer   : {HIDDEN_LAYERS}")
print(f"Total parameters    : {model.count_params()}")

# ============================================================
# TRAINING LOOP
# ============================================================

early_stop = keras.callbacks.EarlyStopping(monitor='loss', mode='min',
                                            patience=PATIENCE, restore_best_weights=True)

print("\n" + "="*60); print("TRAINING"); print("="*60)
history = None
for attempt in range(1, MAX_ATTEMPTS + 1):
    tf.random.set_seed(attempt); np.random.seed(attempt)
    for layer in model.layers:
        if hasattr(layer, 'kernel_initializer'):
            layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))
            layer.bias.assign(layer.bias_initializer(layer.bias.shape))
    history = model.fit(X_train, y_train, epochs=MAX_EPOCHS, verbose=0,
                        callbacks=[early_stop])
    acc_hist = history.history['accuracy']
    best_idx = int(np.argmax(acc_hist)); best_acc = float(acc_hist[best_idx])
    print(f"[Attempt {attempt}] best_epoch={best_idx+1} "
          f"best_acc={best_acc:.4f} last_epoch={len(acc_hist)}")
    if best_acc >= 1.0:
        preds_check = (model.predict(X_train, verbose=0) >= 0.5).astype(int).flatten()
        actual_acc = np.mean(preds_check == y_train.astype(int))
        if actual_acc >= 1.0:
            print(f"Converged at epoch {best_idx+1}.")
            break
        else:
            print(f"[Attempt {attempt}] False convergence — actual acc={actual_acc:.4f}, retrying...")
else:
    raise RuntimeError(f"Training failed after {MAX_ATTEMPTS} attempts.")

preds = (model.predict(X_train, verbose=0) >= 0.5).astype(int).flatten()
print("\nPredictions:")
for xi, yi, pi in zip(X_train, y_train, preds):
    print(f"  x={xi.astype(int)} label={int(yi)} pred={pi} "
          f"{'Yes' if int(yi)==pi else 'No'}")

# >>> VIZ CALL #1: training curve  <<<
plot_training_curve(history, filename="training_curve_multilayer.png")

# ============================================================
# WEIGHT EXTRACTION + AC BUILD
# ============================================================

layers_info    = extract_layer_weights(model)
prev_var_names = [f"x{i+1}" for i in range(n_inputs)]
layer_out_vars = {}
all_acs        = {}
all_nnfs       = {}

print("\n" + "="*60); print("BUILDING ACs"); print("="*60)
for layer_idx, li in enumerate(layers_info):
    layer_name = li['name']
    print(f"\nLayer [{layer_idx+1}]: {layer_name} "
          f"(inputs={li['n_inputs']}, neurons={li['n_neurons']})")
    for j in range(li['n_neurons']):
        ac_j = build_ac_for_neuron(li['W'][:, j], li['b'][j], prev_var_names)
        all_acs[(layer_name, j)] = ac_j
        print(f"  Neuron {j+1}: nodes={ac_j.count}  w={np.round(li['W'][:,j],3)}  b={round(float(li['b'][j]),3)}")
        # >>> VIZ CALL #2: per-neuron AC graph <<<
        visualize_ac(ac_j, f"ac_{layer_name}_n{j+1}",
                     title=f"AC — {layer_name} neuron {j+1}")
    curr_out_vars = [f"l{layer_idx+1}_h{k+1}" for k in range(li['n_neurons'])]
    layer_out_vars[layer_name] = curr_out_vars
    prev_var_names = curr_out_vars

print(f"\nTotal ACs built: {len(all_acs)}")

# ============================================================
# COMPILE AC → NNF
# ============================================================

print("\n" + "="*60); print("COMPILING ACs TO NNF"); print("="*60)
prev_var_names_nnf = [f"x{i+1}" for i in range(n_inputs)]
for layer_idx, li in enumerate(layers_info):
    layer_name = li['name']
    print(f"\nLayer [{layer_idx+1}]: {layer_name}")
    for j in range(li['n_neurons']):
        nnf_j = compile_neuron_obdd_verified(li['W'][:, j], li['b'][j], prev_var_names_nnf)
        all_nnfs[(layer_name, j)] = nnf_j
        all_inputs_j = get_all_binary_inputs(len(prev_var_names_nnf))
        sat = sum(1 for inp in all_inputs_j
                  if eval_nnf(nnf_j, dict(zip(prev_var_names_nnf, inp)),
                              prev_var_names_nnf) == 1)
        print(f"  Neuron {j+1}: NNF nodes={nnf_j.count}  "
              f"satisfying={sat}/{len(all_inputs_j)}")
        # >>> VIZ CALL #3: per-neuron NNF graph <<<
        visualize_nnf(nnf_j, f"nnf_{layer_name}_n{j+1}",
                      title=f"NNF — {layer_name} neuron {j+1}")
    prev_var_names_nnf = layer_out_vars[layer_name]
print("\nAll neurons compiled to NNF via OBDD.")

# ============================================================
# AC vs NNF VERIFICATION
# ============================================================

print("\n" + "="*65); print("FULL INPUT SPACE VERIFICATION"); print("="*65)
prev_vnames_v = [f"x{i+1}" for i in range(n_inputs)]
all_verified = True
for layer_idx, li in enumerate(layers_info):
    layer_name = li['name']
    all_inputs_v = get_all_binary_inputs(len(prev_vnames_v))
    for j in range(li['n_neurons']):
        ac_j, nnf_j = all_acs[(layer_name, j)], all_nnfs[(layer_name, j)]
        mismatches = [(inp, eval_ac(ac_j, dict(zip(prev_vnames_v, inp))),
                       eval_nnf(nnf_j, dict(zip(prev_vnames_v, inp)), prev_vnames_v))
                      for inp in all_inputs_v
                      if eval_ac(ac_j, dict(zip(prev_vnames_v, inp))) !=
                         eval_nnf(nnf_j, dict(zip(prev_vnames_v, inp)), prev_vnames_v)]
        status = "Yes" if not mismatches else f"No — {len(mismatches)} mismatches"
        print(f"  {layer_name} neuron {j+1}: tested {len(all_inputs_v)} → {status}")
        if mismatches: all_verified = False
    prev_vnames_v = layer_out_vars[layer_name]
print("="*65)
print("ALL neurons verified." if all_verified else "Mismatches found.")
print("="*65)

# ============================================================
# TRACEABILITY VERIFICATION
# ============================================================

print("\n" + "="*65); print("TRACEABILITY VERIFICATION"); print("="*65)

def get_binarized_layer_output(model, X, layer_name):
    out = X.copy()
    for layer in model.layers:
        W, b = layer.get_weights()
        z = np.dot(out, W) + b
        if layer.name == layer_name:
            return [int(v > 0) for v in z[0]]
        if 'hidden' in layer.name:
            out = np.array([[float(int(v > 0)) for v in z[0]]])
        else:
            out = 1 / (1 + np.exp(-z))
    return []

W_out, b_out = model.get_layer('output_layer').get_weights()

def binarized_output_forward(binary_prev):
    h = np.array(binary_prev, dtype=np.float32).reshape(1, -1)
    return int((np.dot(h, W_out) + b_out)[0][0] > 0)

all_match = True
for xi in X_train:
    xb = xi.reshape(1, -1)
    print(f"\nInput: {xi.astype(int)}")
    prev_ac_outputs = {f"x{i+1}": float(xi[i]) for i in range(n_inputs)}
    for layer_idx, li in enumerate(layers_info[:-1]):
        layer_name = li['name']
        tf_bin = get_binarized_layer_output(model, xb, layer_name)
        ac_bin = [eval_ac(all_acs[(layer_name, j)], prev_ac_outputs)
                  for j in range(li['n_neurons'])]
        match = tf_bin == ac_bin
        all_match &= match
        print(f"  [{layer_name}] TF={tf_bin}  AC={ac_bin}  Match={'Yes' if match else 'No'}")
        prev_ac_outputs = {layer_out_vars[layer_name][k]: ac_bin[k]
                           for k in range(li['n_neurons'])}
    tf_hid_bin = get_binarized_layer_output(model, xb, layers_info[-2]['name'])
    tf_out = binarized_output_forward(tf_hid_bin)
    ac_out = eval_ac(all_acs[('output_layer', 0)], prev_ac_outputs)
    match_out = tf_out == ac_out
    all_match &= match_out
    print(f"  [output_layer] TF={tf_out}  AC={ac_out}  Match={'Yes' if match_out else 'No'}")
print("\n" + "="*65)
print("ALL LAYERS TRACEABLE." if all_match else "Mismatch detected.")
print("="*65)

# ============================================================
# NNF COMPOSITION
# ============================================================

print("\n" + "="*65); print("NNF COMPOSITION"); print("="*65)
n_hidden_layers = len(layers_info) - 1
current_layer_nnfs = [all_nnfs[(layers_info[0]['name'], j)]
                       for j in range(layers_info[0]['n_neurons'])]
for layer_idx in range(1, len(layers_info)):
    layer_name = layers_info[layer_idx]['name']
    next_layer_nnfs = [all_nnfs[(layer_name, j)]
                       for j in range(layers_info[layer_idx]['n_neurons'])]
    composed_layer_nnfs = []
    for nnf_j in next_layer_nnfs:
        c = compose_nnfs(nnf_j, current_layer_nnfs, n_inputs)
        c.n_vars = n_inputs
        composed_layer_nnfs.append(c)
    current_layer_nnfs = composed_layer_nnfs
    print(f"  After composing layer [{layer_idx+1}] ({layer_name}): "
          f"{len(current_layer_nnfs)} NNF(s), nodes={[c.count for c in current_layer_nnfs]}")
composed_nnf = current_layer_nnfs[0]
composed_nnf.n_vars = n_inputs
print(f"\n  Final composed NNF nodes : {composed_nnf.count}")

# >>> VIZ CALL #4: composed end-to-end NNF <<<
visualize_nnf(composed_nnf, "nnf_composed_end_to_end",
              title=f"Composed end-to-end NNF ({composed_nnf.count} nodes)")

print("\n  Verifying composed NNF vs binarized TF:")
all_inputs_orig = get_all_binary_inputs(n_inputs)
compose_ok = True
for inp in all_inputs_orig:
    vals = {f"x{i+1}": inp[i] for i in range(n_inputs)}
    composed_out = eval_nnf(composed_nnf, vals, [f"x{i+1}" for i in range(n_inputs)])
    xb = np.array(inp, dtype=np.float32).reshape(1,-1)
    tf_hid_bin = get_binarized_layer_output(model, xb, layers_info[-2]['name'])
    tf_ref = binarized_output_forward(tf_hid_bin)
    status = "Yes" if composed_out == tf_ref else "No"
    if composed_out != tf_ref: compose_ok = False
    print(f"    x={inp}  composed_NNF={composed_out}  binarized_TF={tf_ref}  {status}")
print("\nComposed NNF correct." if compose_ok else "Composed NNF has errors.")
print("End-to-end NNF composition complete.")

# ============================================================
# ROBUSTNESS ANALYSIS
# ============================================================

print("\n" + "="*65); print("MODEL-BASED ROBUSTNESS"); print("="*65)
var_names_orig = [f"x{i+1}" for i in range(n_inputs)]
all_inputs_orig = get_all_binary_inputs(n_inputs)
print("\n  1-flip sensitivity:")
for inp in all_inputs_orig:
    vals = {f"x{i+1}": inp[i] for i in range(n_inputs)}
    out = eval_nnf(composed_nnf, vals, var_names_orig)
    sensitive = []
    for k in range(n_inputs):
        flipped = inp.copy(); flipped[k] = 1 - flipped[k]
        flipped_vals = {f"x{i+1}": flipped[i] for i in range(n_inputs)}
        if eval_nnf(composed_nnf, flipped_vals, var_names_orig) != out:
            sensitive.append(f"x{k+1}")
    print(f"    x={inp}  out={out}  sensitive_to={sensitive}")

positive_inputs = [inp for inp in all_inputs_orig
                   if eval_nnf(composed_nnf,
                               {f"x{i+1}": inp[i] for i in range(n_inputs)},
                               var_names_orig) == 1]
n_pos = len(positive_inputs); n_tot = len(all_inputs_orig)
print(f"\n  Model count : {n_pos} / {n_tot}")
print("\n  Prime Implicant Explanations (output=1):")
for inp in positive_inputs:
    print(f"    PI: { {f'x{i+1}': inp[i] for i in range(n_inputs)} }")

n = n_inputs; total = 2 ** n; M = 0
print(f"\n  Minimum Hamming distance to nearest negative input:")
negative_inputs = [inp for inp in all_inputs_orig if inp not in positive_inputs]
for inp in positive_inputs:
    min_dist = n
    for neg in negative_inputs:
        dist = sum(1 for a, b in zip(inp, neg) if a != b)
        min_dist = min(min_dist, dist)
    M += min_dist
    print(f"    x={inp}  min_hamming_to_negative={min_dist}")
mrf = round(M / total, 4)
print(f"\n  Robustness weighted sum M    = {M}")
print(f"  Total inputs (2^n)           = {total}")
print(f"  Model-Based Robustness mr(f) = {mrf}")
print(f"  Positive inputs              : {n_pos} / {total}")
print("="*65)

# ============================================================
# FINAL VIZ SUMMARY
# ============================================================
print("\n" + "="*65); print("VISUALIZATION SUMMARY"); print("="*65)
written = sorted(os.listdir(VIZ_DIR))
if written:
    print(f"Wrote {len(written)} files to {os.path.abspath(VIZ_DIR)}/:")
    for f in written:
        size = os.path.getsize(os.path.join(VIZ_DIR, f))
        print(f"  {f}  ({size:,} bytes)")
else:
    print(f"WARNING: {VIZ_DIR}/ is empty.")
    print("Install networkx if missing:  pip install networkx")
print("="*65)

print("\nScaled pipeline complete.")

[VIZ] output folder: /content/viz_output
NN2Logic — Scaled Network Experiment

Dataset      : xor
Inputs       : 2
Samples      : 4
Hidden layers: [8, 4]


Model: "nn2logic_scaled"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer_1 (Dense)          │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer_2 (Dense)          │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65 (260.00 B)

 Trainable params: 65 (260.00 B)

 Non-trainable params: 0 (0.00 B)


Total hidden layers : 2
Neurons per layer   : [8, 4]
Total parameters    : 65

TRAINING
[Attempt 1] best_epoch=19 best_acc=1.0000 last_epoch=10000
Converged at epoch 19.

Predictions:
  x=[0 0] label=0 pred=0 Yes
  x=[0 1] label=1 pred=1 Yes
  x=[1 0] label=1 pred=1 Yes
  x=[1 1] label=0 pred=0 Yes
[VIZ] wrote viz_output/training_curve_multilayer.png

BUILDING ACs

Layer [1]: hidden_layer_1 (inputs=2, neurons=8)
  Neuron 1: nodes=9  w=[ 2.465 -1.429]  b=0.521
  Neuron 2: nodes=9  w=[-0.84  -0.095]  b=0.317
  Neuron 3: nodes=9  w=[-0.678  2.074]  b=-0.083
  Neuron 4: nodes=9  w=[ 0.307 -1.496]  b=0.234
  Neuron 5: nodes=9  w=[-0.51  -0.575]  b=0.589
  Neuron 6: nodes=9  w=[1.963 1.843]  b=-0.388
  Neuron 7: nodes=9  w=[ 0.655 -0.153]  b=0.139
  Neuron 8: nodes=9  w=[-2.143 -2.09 ]  b=0.52

Layer [2]: hidden_layer_2 (inputs=8, neurons=4)
  Neuron 1: nodes=27  w=[ 1.216 -0.523 -0.214 -0.469 -0.029 -0.218  0.88   0.472]  b=0.347
  Neuron 2: nodes=27  w=[ 1.651 -0.129  1.432 -1.193 -0.782 

In [4]:
# ============================================================
# ZIP ALL VISUALIZATION FILES FOR EASY DOWNLOAD
# ============================================================

import shutil
from google.colab import files

zip_name = "nn2logic_visualizations"

# Creates: nn2logic_visualizations.zip
shutil.make_archive(zip_name, 'zip', VIZ_DIR)

print(f"\nCreated ZIP file: {zip_name}.zip")

# Auto download
files.download(f"{zip_name}.zip")


Created ZIP file: nn2logic_visualizations.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>